# COMP5339 Assignment 1 — Electricity Sector Data Integration

**Team:** [Name A] · [Name B] · [Name C]  
**Stage:** 1 (Week 8)

This notebook orchestrates the full data pipeline: acquiring electricity sector data from NGER, CER, and ABS; integrating, cleaning, and augmenting it with geocoded facility locations; and loading it into DuckDB for analysis.

Implementation modules live under `pipeline/`; this notebook is the orchestration and documentation layer.

## How to run

1. Ensure Python 3.10+ is installed.
2. Run the **Setup** cell below once to install dependencies.
3. Review the **Configuration** cell — toggle `USE_CACHED_*` flags as needed:
   - `True` (default) — use the cached raw data shipped with the submission. Runs fast, no network required.
   - `False` — fetch fresh data from each source. Required for a true cold-run demonstration; takes a few minutes (mostly geocoding).
4. **Kernel → Restart & Run All**.

All intermediate data is written under `data/` and the final database to `warehouse.duckdb`.

In [ ]:
# %%capture
# Install project dependencies.
# Output is suppressed; remove the %%capture line to see pip output.
# %pip install -r requirements.txt

## Configuration

Flip flags to `False` to force a fresh fetch from the respective source. Recommended: leave `USE_CACHED_GEO = True` after the first run — Nominatim is rate-limited to 1 request/second and re-geocoding takes several minutes.

In [1]:
# ── Pipeline configuration ─────────────────────────────────────
USE_CACHED_NGER = True   # NGER electricity sector data 
USE_CACHED_CER  = True   # CER large-scale renewables (csv download)
USE_CACHED_ABS  = True   # ABS Data by Regions (XLSX download)
USE_CACHED_GEO  = True   # Geocoded facility coordinates (Nominatim)
# ───────────────────────────────────────────────────────────────

In [2]:
from pathlib import Path

import pandas as pd

# Make the local `pipeline/` package importable regardless of the
# notebook's launch directory.
import sys
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure data directories exist up-front.
for sub in ("raw", "cleaned", "curated"):
    (PROJECT_ROOT / "data" / sub).mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

---

## 1. Data Acquisition

Three sources are retrieved programmatically. Each source has a dedicated module under `pipeline/acquire/` (one file per source). Raw payloads are saved to `data/raw/<source>/` in their native format (JSON for API responses, XLSX for file downloads).

### 1.1 NGER — Electricity sector emissions and generation

Ten annual datasets from the Clean Energy Regulator's public data API, spanning financial years 2014-15 through 2023-24. Each endpoint returns one year's facility-level emissions and generation records as JSON.

The reporting year is not present in the payload, so we inject it per-row at acquisition time (record provenance, not transformation).

In [3]:
from pipeline.acquire.nger import fetch_nger

fetch_nger(use_cache=USE_CACHED_NGER)

[NGER-acquire] Acquiring 10 datasets, FY2014-15 → FY2023-24
[NGER-acquire]   FY2014-15 (ID0075) → 424 rows [cached]
[NGER-acquire]   FY2015-16 (ID0076) → 480 rows [cached]
[NGER-acquire]   FY2016-17 (ID0077) → 486 rows [cached]
[NGER-acquire]   FY2017-18 (ID0078) → 522 rows [cached]
[NGER-acquire]   FY2018-19 (ID0079) → 583 rows [cached]
[NGER-acquire]   FY2019-20 (ID0080) → 621 rows [cached]
[NGER-acquire]   FY2020-21 (ID0081) → 655 rows [cached]
[NGER-acquire]   FY2021-22 (ID0082) → 691 rows [cached]
[NGER-acquire]   FY2022-23 (ID0083) → 705 rows [cached]
[NGER-acquire]   FY2023-24 (ID0243) → 775 rows [cached]
[NGER-acquire] Total: 5,942 rows (0 fetched, 10 cached)


<cell_type>markdown</cell_type>### 1.2 CER — Large-scale renewable power stations

Four datasets from the Clean Energy Regulator's public website: `historical_accredited` (all historical accreditations), `approved_2026` (accreditations active for the 2026 reporting year), `committed` (projects with financial/grid commitments), and `probable` (speculative pipeline).

Each is served as a CSV behind a `/document/<slug>` URL on one of two landing pages. Rather than hardcode the slugs, the acquisition module fetches the page HTML, parses anchor `href`s, and searches for link slugs matching keyword rules (`must_contain` / `must_not_contain`). For the historical page, which exposes both XLSX and CSV variants of the same file, we HEAD each candidate and keep the one that serves a CSV content-type.

In [ ]:
from pipeline.acquire.cer import fetch_cer

fetch_cer(use_cache=USE_CACHED_CER)

<cell_type>markdown</cell_type>### 1.3 ABS — Data by Region (Population & People)

One XLSX downloaded from the ABS 2011-24 Data by Region methodology page. The page hosts ten topic workbooks with opaque filenames (`14100DO0001_2011-24.xlsx`), so we scrape the page HTML, find the heading that contains `population` + `people`, and pick the first XLSX link that follows it. This keeps the acquisition robust against ABS renumbering the file IDs in future releases.

In [ ]:
from pipeline.acquire.abs import fetch_abs

fetch_abs(use_cache=USE_CACHED_ABS)

<cell_type>markdown</cell_type>---

## 2. Data Cleaning

Per-source cleaning modules under `pipeline/clean/`. Each loads its raw files, normalises column names (lowercasing + a single rename map that collapses every variant seen across years/sources), coerces types, and writes a tidy CSV under `data/cleaned/`.

### 2.1 NGER — Normalise column drift across 10 annual endpoints

The API returns JSON with at least four different column-naming conventions over the decade (camelCase vs lowercase, `controllingcorporation` vs `reportingEntity` in 2015-16, trailing-`2` typos in 2016-17). We handle the drift with a single rename map applied after lowercasing, rather than per-year branching.

In [4]:
from pipeline.clean.nger import clean_nger

df = clean_nger()

df.head()

[NGER-clean] Loading 10 years from data/raw/nger/
[NGER-clean] Concatenated 5,942 rows
[NGER-clean] Row types: {'F': 4862, 'C': 1056, 'FA': 18, '-': 4, '': 2}
[NGER-clean] JV double-count flagged: 30 rows
[NGER-clean] Missing state: 905 (expected ≈ corporate-total rows: 1,056)
[NGER-clean] Saved to data/cleaned/nger.csv (0.72 MB)


,reporting_year,reporting_entity,facility_name,row_type,state,grid_connected,grid,primary_fuel,electricity_production_gj,electricity_production_mwh,scope1_emissions_tco2e,scope2_emissions_tco2e,total_emissions_tco2e,emission_intensity_tco2e_per_mwh,jv_double_counted,important_notes
0,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Gunning Wind Farm,F,NSW,On,NEM,Wind,567719.0,157700.0,19.0,293.0,312,0.0,False,N/A
1,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Royalla Solar Farm,F,ACT,On,NEM,Solar,213115.0,59199.0,0.0,15.0,15,0.0,False,N/A
2,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Waubra Wind Farm,F,VIC,On,NEM,Wind,2461803.0,683834.0,77.0,1144.0,1221,0.0,False,N/A
3,2014-15,ACCIONA ENERGY OCEANIA PTY LTD,Corporate Total,C,N/A,N/A,N/A,N/A,3242637.0,900733.0,96.0,1452.0,1548,NaN,False,N/A
4,2014-15,AGL ENERGY LIMITED,Banimboola Hydro,F,VIC,On,NEM,Hydro,137094.0,38082.0,5.0,17.0,22,0.0,False,N/A


In [5]:
df.facility_name.value_counts(dropna=False)

facility_name
Corporate Total                                 1056
Facility                                          17
Daandine Power Station                            14
Royalla Solar Farm                                11
Bayswater Power Station                           11
                                                ... 
Stanhope Solar Project No. 5 Pty Ltd               1
Stanhope Solar Project Pty Ltd                     1
Wunghnu Project Pty Ltd                            1
Tailem Bend 2 Hybrid Renewable Power Station       1
Wandoan Solar Farm 1                               1
Name: count, Length: 857, dtype: int64

<cell_type>markdown</cell_type>---

## 3. Data Integration

Three curated tables, all keyed for easy joining:

- **`facility`** — one row per unique physical facility. Built by matching NGER `facility_name` to CER `station_name` using a three-key conjunction: normalised name **and** state **and** fuel category. Fuel category mapping collapses NGER `Landfill Gas` / `Waste Coal Mine Gas` → CER `Biomass`. Match algorithm: exact normalised name → rapidfuzz `token_set_ratio ≥ 85` → `nger_only` fallthrough. Unclaimed CER stations → `cer_only`. All CER columns preserved; `lat`/`lon` placeholders for Task 4.
- **`generation`** — one row per facility-year (NGER `F` and `FA` rows only; corporate roll-ups `C`, JV double-counts, and sentinel row-types dropped at load). All 16 NGER columns preserved, plus `facility_id` FK and derived `financial_year_end` (int: `"2023-24"` → `2024`).
- **`abs_population`** — the cleaned narrow ABS table (Estimated Resident Population and demographic companions) plus a 2-letter `state` abbrev derived from the ASGS code's leading digit, so cross-source joins work at state level without a separate lookup table. ERP is published on a "year ended 30 June" basis, which aligns 1:1 with the Australian NGER reporting FY (`FY 2023-24` ↔ ABS `year == 2024`).

All three share `state` as a 2-letter abbrev. Filter `abs_population.geography_level = 'STATE'` for 9 rows per year that join 1:1 to `facility.state` and `generation.state`. ERP is a natural denominator for per-capita emissions and per-capita renewable capacity.

In [ ]:
from pipeline.integrate.facility_generation import integrate_facility_generation
from pipeline.integrate.abs_population import integrate_abs_population

facility, generation = integrate_facility_generation()
abs_population = integrate_abs_population()

In [ ]:
# Match outcome summary + smoke-test cross-source join.
print("Match mix in facility table:")
print(facility["match_status"].value_counts().to_string())
print()

print("Cross-source join smoke test (NSW, FY ending 2023):")
nsw_abs = abs_population.query(
    "geography_level == 'STATE' and state == 'NSW' and year == 2023"
).iloc[0]
nsw_gen = generation.query("state == 'NSW' and financial_year_end == 2023")
print(f"  ABS NSW estimated resident population (30 Jun 2023): {int(nsw_abs['estimated_resident_population_no']):,}")
print(f"  NGER NSW facilities reporting in FY23: {nsw_gen['facility_id'].nunique():,}")
print(f"  NGER NSW total emissions FY23 (tCO2e): {nsw_gen['total_emissions_tco2e'].sum():,.0f}")
print(f"  NSW per-capita emissions (kg CO2e / person): {nsw_gen['total_emissions_tco2e'].sum() * 1000 / nsw_abs['estimated_resident_population_no']:,.1f}")

In [ ]:
from pipeline.clean.cer import clean_cer

cer_df = clean_cer()

cer_df.head()

### 2.3 ABS — Clean Table 1 into a narrow CSV

Only **Table 1** (ASGS regions: AUS → State → GCCSA → SA4 → SA3 → SA2) is kept; Table 2 (LGAs, 550 codes) is a parallel geography that neither NGER nor CER use natively, so carrying it forward would force an unnecessary postcode/LGA concordance.

Column scope is deliberately narrow: we keep only the 9 columns sitting under the *"Estimated resident population - year ended 30 June"* banner (ERP total, population density, ERP by sex, median age M/F/persons, working-age count and %). The remaining 150-odd columns on Table 1 cover unrelated themes (births/deaths, migration, ATSI, citizenship, language, ADF service) and are dropped.

The sheet has five metadata rows above the real header (row 6), two footnote rows at the bottom, and `-` as the suppressed-value sentinel. After dropping metadata/footer, slicing to the banner, slugifying the headers, and coercing dtypes, we derive a `geography_level` column from the Code pattern (`AUS`, `1` → `STATE`, `1GSYD` → `GCCSA`, 3/5/9-digit → `SA4`/`SA3`/`SA2`) so downstream queries can filter by level without regex.

In [ ]:
from pipeline.clean.abs import clean_abs

abs_df = clean_abs()

abs_df[["code", "label", "year", "geography_level"]].head()

---

## 3. Data Integration

*To be implemented.*

---

## 4. Data Augmentation — Geocoding (Nominatim)

We augment the `facility` table with geographic coordinates by querying the Nominatim public geocoding API (OpenStreetMap). Free, no API key, complies with Nominatim's usage policy: custom User-Agent identifying the assignment + a contact email, and ≤1 request/second (1.1 s sleep between live calls).

**Two-tier query strategy** per facility:
1. **Precise** — `"<facility_name>, <postcode>, <state>, Australia"` with `countrycodes=au`. Hits when the facility itself is tagged in OSM (e.g. *Waubra Wind Farm*, *Bogong Power Station*).
2. **Postcode-centroid fallback** (only on tier-1 miss) — `"<postcode>, Australia"`. Always resolves for a valid AU postcode and gives a suburb-level location for facilities OSM doesn't carry by name.

**Caching** — every query (precise OR fallback) is cached separately under its literal query string in `data/raw/geocode/nominatim_cache.json`. Value carries `lat`, `lon`, `name`, `state`, `postcode`, `area` (Nominatim display name), and `source` (`"facility"` or `"postcode"`). `null` value means a definitive miss — also cached so we don't refetch. HTTP errors are *not* cached so they retry on the next run. Cache is rewritten after every successful call; the 1.1 s sleep absorbs the disk I/O.

**Scope** — 3,432 rows flagged `use_for_geocoding == True` (the CER `historical_accredited` set, the only CER subset with both a stable identity and a postcode). With cache populated, the call below is a no-op (~1 s to load the JSON, no network); a cold run is ~3 hours due to Nominatim's per-request latency on top of the 1 req/s rate limit.

In [ ]:
from pipeline.augment.geocode import geocode_facilities

# Resumable: skips any query already in the JSON cache. Cold run
# is ~3 h; cached run is ~1 s. Re-run any time to retry the
# uncached entries (e.g. queries that hit transient DNS errors).
cache = geocode_facilities()

### 4.1 Coverage breakdown

How many facilities resolved precisely (their own OSM record), how many fell back to a postcode centroid, and how many failed entirely. Failures are mostly small commercial-rooftop solar (median 0.34 MW across the CER-only majority) — not in any address database because they're literal panels on warehouse/business roofs.

In [ ]:
import json
from pathlib import Path

cache = json.loads(Path("data/raw/geocode/nominatim_cache.json").read_text())
precise  = sum(1 for v in cache.values() if v and v["source"] == "facility")
fallback = sum(1 for v in cache.values() if v and v["source"] == "postcode")
miss     = sum(1 for v in cache.values() if v is None)

print(f"Cache entries:       {len(cache):,}")
print(f"  Precise hits:      {precise:,}")
print(f"  Fallback hits:     {fallback:,}")
print(f"  Definitive misses: {miss:,}")

---

## 5. Storage — DuckDB schema and load

*To be implemented.*

---

## 6. Exploration and visualisation

*To be implemented.*